# Step 1: Load and inspect raw ETF files

We load the three Kaggle price files (SPY, TLT, XLF), convert the `Date` column to real dates using a strict format so any malformed date stops the notebook, and sort each table by date. We then check for repeated dates and record each file's first date, last date and number of rows. These checks confirm the files cover the expected period (25 Feb 2005 – 10 Nov 2017) before anything is built on them.

In [26]:
RAW = 'raw/' # relative to the notebook's working folder
import os; print(os.listdir(RAW)) # should list the 4 files

['VIXCLS.csv', 'tlt.us.txt', 'spy.us.txt', 'xlf.us.txt']


In [27]:
import os
import pandas as pd

FILES = {"SPY": "spy.us.txt", "TLT": "tlt.us.txt", "XLF": "xlf.us.txt"}

def load_etf(path):
    df = pd.read_csv(path)

    # strict: raises on bad dates, so pandas stops with an error instead of carrying on.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
    df = df.sort_values("Date", kind="stable").reset_index(drop=True)

    return df

def summarise(ticker, df):
    dup_dates = df.loc[df["Date"].duplicated(keep=False), "Date"].unique()
    return {"ticker": ticker,
            "start": df["Date"].min().date(),
            "end": df["Date"].max().date(),
            "rows": len(df),
            "n_dup_dates": len(dup_dates)}

data = {t: load_etf(os.path.join(RAW, f)) for t, f in FILES.items()}
summary_df = pd.DataFrame([summarise(t, df) for t, df in data.items()]).set_index("ticker")
summary_df

,start,end,rows,n_dup_dates
ticker,,,,
SPY,2005-02-25,2017-11-10,3201,0
TLT,2005-02-25,2017-11-10,3201,0
XLF,2005-02-25,2017-11-10,3201,0


# Step 2: Clean and sanity-check prices

We drop the `OpenInt` column after confirming it is all zeros, since open interest has no meaning for an ETF. We then flag rows that are physically impossible: a High below the Low, a price of zero or less, or negative volume. These rows are reported, not deleted. Genuine extreme days, such as the 2010 flash crash, are kept, because they are exactly the high-volatility events we want to forecast.

In [28]:
PRICE_COLS = ["Open", "High", "Low", "Close"]

def clean_and_check(ticker, df):
    # 1. OpenInt: confirm it carries no information, then drop it
    assert (df["OpenInt"] == 0).all(), f"[{ticker}] OpenInt has non-zero values"
    df = df.drop(columns="OpenInt")

    # 2. Sanity checks: one boolean mask per rule (True = impossible row)
    checks = {
        "high_below_low": df["High"] < df["Low"],
        "price_not_positive": (df[PRICE_COLS] <= 0).any(axis=1),
        "negative_volume": df["Volume"] < 0,
    }

    # 3. Report, don't delete: print how many rows break each rule
    for rule, mask in checks.items():
        n_bad = int(mask.sum())
        print(f"[{ticker}] {rule}: {n_bad} rows")
        if n_bad:
            print("   dates:", df.loc[mask, "Date"].dt.strftime("%Y-%m-%d").tolist())

    return df

# Apply to the three tables from step 1
data = {t: clean_and_check(t, df) for t, df in data.items()}

[SPY] high_below_low: 0 rows
[SPY] price_not_positive: 0 rows
[SPY] negative_volume: 0 rows
[TLT] high_below_low: 0 rows
[TLT] price_not_positive: 0 rows
[TLT] negative_volume: 0 rows
[XLF] high_below_low: 0 rows
[XLF] price_not_positive: 0 rows
[XLF] negative_volume: 0 rows


# Step 3: Merge VIX and align to SPY trading days

We add the VIX (series VIXCLS from FRED) and combine all four files into one table with one row per SPY trading day. Where TLT, XLF or the VIX has no value for a SPY day, we carry the last known value forward, never backward, so no row ever uses information from the future. A before/after table shows the percentage of missing values in each column.

In [29]:
# 3a. Load the VIX from FRED; holidays are "." or blank, so read them as missing
vix = pd.read_csv(RAW + "VIXCLS.csv", na_values=".")
vix = vix.rename(columns={vix.columns[0]: "Date", "VIXCLS": "VIX"})  # date column name varies by download
vix["Date"] = pd.to_datetime(vix["Date"], format="%Y-%m-%d")

# 3b. Prefix TLT/XLF columns so they don't clash with SPY's (e.g. "TLT_Close")
def add_prefix(df, ticker):
    return df.rename(columns={c: f"{ticker}_{c}" for c in df.columns if c != "Date"})

tlt = add_prefix(data["TLT"], "TLT")
xlf = add_prefix(data["XLF"], "XLF")

# 3c. Start from SPY's dates and left-join the others onto them
merged = data["SPY"]
for other in [tlt, xlf, vix]:
    merged = merged.merge(other, on="Date", how="left", validate="one_to_one")
assert len(merged) == len(data["SPY"]), "Merge changed the number of SPY rows"

# 3d. % missing per column BEFORE filling
missing_before = merged.isna().mean() * 100

# 3e. Forward-fill only (never backward): copy the last known value onto gaps
fill_cols = [c for c in merged.columns if c.startswith(("TLT_", "XLF_")) or c == "VIX"]
merged[fill_cols] = merged[fill_cols].ffill()

# 3f. % missing AFTER, side by side with before
missing_after = merged.isna().mean() * 100
missing_table = pd.DataFrame({"before_%": missing_before, "after_%": missing_after}).round(2)
missing_table

,before_%,after_%
Date,0.0,0.0
Open,0.0,0.0
High,0.0,0.0
Low,0.0,0.0
Close,0.0,0.0
Volume,0.0,0.0
TLT_Open,0.0,0.0
TLT_High,0.0,0.0
TLT_Low,0.0,0.0
TLT_Close,0.0,0.0


# Step 4 — Compute returns, target and memory features (1–3)

We compute SPY's daily log returns and, from them, realised volatility (RV) over 1, 5 and 22 trading days: yesterday's shock, last week and last month. These three features form the HAR-RV benchmark. We also build the target: the log of realised volatility over the *next* 5 trading days. On 11 days (10 in training, 1 in test) SPY's close repeats exactly, giving a zero return whose log is −∞. We raise these to a floor equal to the smallest non-zero absolute return in the training period, computed on training data only, so no test information is used and no genuine trading day is altered. A final check confirms that only the zero-return days were changed.

In [30]:
import numpy as np

TRAIN_END = "2014-12-31"   # last training day (plan: train 2005-2014)

def log_returns(close):
    """Daily log return: r_t = ln(Close_t / Close_{t-1}). First row is NaN."""
    return np.log(close).diff()

def log_rv(r, n, floor):
    """Log realised volatility over the n days UP TO AND INCLUDING t (usable as a feature).
    RV = sqrt(sum of squared returns); n=1 gives |r_t|. Values below `floor` are raised to it."""
    rv = np.sqrt(r.pow(2).rolling(n).sum())
    return np.log(rv.clip(lower=floor))

def log_frv(r, n, floor):
    """Log realised volatility over the n days AFTER t (target only, never a feature)."""
    return log_rv(r, n, floor).shift(-n)

# 4a. SPY daily log returns
merged["r"] = log_returns(merged["Close"])

# 4b. Zero-return check: days where the close repeats exactly (prices have only 2-3 decimals)
is_zero  = merged["r"] == 0
is_train = merged["Date"] <= TRAIN_END
print(f"Zero-return days: {is_zero.sum()} (train: {(is_zero & is_train).sum()}, test: {(is_zero & ~is_train).sum()})")
print(merged.loc[is_zero, "Date"].dt.strftime("%Y-%m-%d").tolist())

# 4c. Floor = smallest NON-ZERO |r| in the training period (training only -> no test information).
#     Zero days get the smallest move we can actually measure; no real day is changed.
FLOOR = merged.loc[is_train & ~is_zero, "r"].abs().min()
print(f"Floor: {FLOOR:.2e} (log = {np.log(FLOOR):.2f})")

# 4d. Target: log RV over the next 5 trading days
merged["y"] = log_frv(merged["r"], 5, FLOOR)

# 4e. Features 1-3 (HAR memory): yesterday's shock, last week, last month
merged["f1"] = log_rv(merged["r"], 1, FLOOR)    # log RV_1 = log|r_t|
merged["f2"] = log_rv(merged["r"], 5, FLOOR)    # log RV_5
merged["f3"] = log_rv(merged["r"], 22, FLOOR)   # log RV_22

# 4f. Sanity check: the floor changed only the zero-return days
with np.errstate(divide="ignore"):
    changed = (merged["f1"] != np.log(merged["r"].abs())) & merged["r"].notna()
assert changed.sum() == is_zero.sum(), "Floor altered non-zero days"

Zero-return days: 11 (train: 10, test: 1)
['2005-08-22', '2005-11-08', '2007-12-19', '2012-07-30', '2012-10-31', '2013-03-27', '2013-06-10', '2013-07-30', '2013-10-21', '2014-02-26', '2017-01-10']
Floor: 3.27e-05 (log = -10.33)


# Step 5 — Build the leverage feature (5)

We add SPY's 1-day log return with its sign kept (f5 = rₜ). Feature 1 already measures the size of today's move; this feature adds its direction, because falls tend to raise future volatility more than rises of the same size (the leverage effect). It uses only day t's return, so it is known at time t and leaks nothing from the future. A quick check on the training period confirms the effect in our data: after the 10% biggest falls, next-week log volatility averages −3.44, against −3.70 after the 10% biggest rises and −4.03 overall.

In [31]:
# 5a. Feature 5 (leverage effect): today's return WITH its sign.
#     f1 already holds the SIZE of today's move (log|r_t|); f5 adds its DIRECTION.
merged["f5"] = merged["r"]

# 5b. Sanity check: f5 uses only day t's return (no shift), so it is known at t -> no leakage
assert merged["f5"].equals(merged["r"]), "f5 must equal today's return"

# 5c. Leverage-effect check, on TRAINING data only (the test set stays untouched until Sat 14 Nov):
#     average next-week log volatility after the biggest falls vs the biggest rises
train = merged[merged["Date"] <= TRAIN_END]
low, high = train["f5"].quantile([0.10, 0.90])
print(f"Mean y after the 10% biggest falls: {train.loc[train['f5'] <= low,  'y'].mean():.2f}")
print(f"Mean y after the 10% biggest rises: {train.loc[train['f5'] >= high, 'y'].mean():.2f}")
print(f"Mean y on all training days:        {train['y'].mean():.2f}")

Mean y after the 10% biggest falls: -3.44
Mean y after the 10% biggest rises: -3.70
Mean y on all training days:        -4.03


# Step 6 — Build VIX and cross-asset features (4, 6–8)

We add four features from outside SPY. Feature 4 is the log of the VIX, the market's own 30-day volatility forecast. Feature 6 is TLT's log realised volatility over the last 22 days (bond-market turbulence). Feature 7 is XLF's 22-day log realised volatility minus SPY's (f3), which measures how much more turbulent banks are than the whole market. Feature 8 is TLT's 22-day log return, capturing money moving into bonds (flight to safety). Features 6–7 reuse the realised-volatility function from Step 4, and all four use only data up to day t. The first 22 rows are missing for features 6–8 and are removed in Step 8.

In [32]:
# 6a. Feature 4: log VIX, the market's own 30-day volatility forecast.
#     VIXCLS is the closing value on day t, so it is known at t.
assert (merged["VIX"] > 0).all(), "VIX must be positive before taking its log"
merged["f4"] = np.log(merged["VIX"])

# 6b. Daily log returns for the two other ETFs (same function as SPY in step 4)
r_tlt = log_returns(merged["TLT_Close"])
r_xlf = log_returns(merged["XLF_Close"])

# 6c. Feature 6: bond-market turbulence = TLT log RV over the last 22 days
merged["f6"] = log_rv(r_tlt, 22, FLOOR)

# 6d. Feature 7: banking stress relative to the whole market
#     = XLF log RV_22 minus SPY log RV_22 (f3). A difference of logs = log of the ratio,
#     so 0 means "banks as turbulent as the market", 0.69 means "twice as turbulent".
merged["f7"] = log_rv(r_xlf, 22, FLOOR) - merged["f3"]

# 6e. Feature 8: flight to safety = TLT log return over the last 22 days
merged["f8"] = np.log(merged["TLT_Close"] / merged["TLT_Close"].shift(22))

# 6f. Sanity checks: only the first 22 rows may be missing (step 8 trims them)
step6 = ["f4", "f6", "f7", "f8"]
print(merged[step6].isna().sum())
print(merged[step6].describe().round(3))

f4     0
f6    22
f7    22
f8    22
dtype: int64
             f4        f6        f7        f8
count  3201.000  3179.000  3179.000  3179.000
mean      2.855    -3.310     0.364     0.004
std       0.387     0.356     0.255     0.039
min       2.213    -4.475    -0.236    -0.153
25%       2.561    -3.559     0.187    -0.020
50%       2.773    -3.352     0.314     0.003
75%       3.084    -3.061     0.517     0.026
max       4.393    -2.232     1.332     0.245


# Step 7 — Build activity and trend features (9–10)

We add two features from SPY's own trading. Feature 9 is the log of today's volume divided by the average volume over the previous 22 trading days, so 0 means a normal day and positive values mean unusually busy trading; today is excluded from the average so that a spike day does not dampen its own reading. Feature 10 is SPY's log return over the last 22 trading days, capturing the market's recent trend. Both use only data up to day t. The first 22 rows are missing and are removed in Step 8.

In [33]:
# 7a. Feature 9: unusual trading activity = log(today's volume / a normal day's volume).
#     "Normal" = mean volume over the 22 trading days BEFORE today (shift(1) excludes today,
#     so a spike day doesn't inflate its own baseline).
assert (merged["Volume"] > 0).all(), "Volume must be positive before taking a log"
normal_volume = merged["Volume"].rolling(22).mean().shift(1)
merged["f9"] = np.log(merged["Volume"] / normal_volume)

# 7b. Feature 10: market trend = SPY log return over the last 22 trading days
merged["f10"] = np.log(merged["Close"] / merged["Close"].shift(22))

# 7c. Sanity checks: only the first 22 rows may be missing (step 8 trims them)
step7 = ["f9", "f10"]
print(merged[step7].isna().sum())
print(merged[step7].describe().round(3))

f9     22
f10    22
dtype: int64
             f9       f10
count  3179.000  3179.000
mean     -0.045     0.006
std       0.344     0.046
min      -1.826    -0.364
25%      -0.262    -0.012
50%      -0.056     0.013
75%       0.159     0.032
max       1.493     0.195


# Step 8 — Trim incomplete rows, split and save model table

We remove the 22 rows at the start, where the 22-day features cannot yet be computed, and the 5 rows at the end, where the 5-day-ahead target does not exist. Each remaining row is labelled train (2005–2014) or test (2015 onwards). We then drop the last 5 training rows (24–31 Dec 2014), the embargo: their targets are built from returns in early January 2015, which belong to the test period. The final table has 2,452 training rows (30 Mar 2005 – 23 Dec 2014) and 717 test rows (2 Jan 2015 – 3 Nov 2017), and is saved as `data/processed/model_table.csv`, the single table every model uses.

In [34]:
import os

FEATURES = [f"f{i}" for i in range(1, 11)]   # f1 ... f10
EMBARGO = 5                                    # target looks 5 trading days ahead

# 8a. Remove incomplete rows: the first ~22 (22-day features not yet computable)
#     and the last 5 (no 5-day-ahead target yet)
complete = merged.dropna(subset=["y"] + FEATURES).copy()
print(f"Dropped {len(merged) - len(complete)} incomplete rows: "
      f"{(merged['Date'] < complete['Date'].min()).sum()} at the start, "
      f"{(merged['Date'] > complete['Date'].max()).sum()} at the end")

# 8b. Split column: train = 2005-2014, test = 2015 onwards
complete["split"] = np.where(complete["Date"] <= TRAIN_END, "train", "test")

# 8c. Embargo: drop the last 5 training rows. Their target (next 5 days' volatility)
#     is built from returns in early January 2015, i.e. from test-period days.
embargo_idx = complete.index[complete["split"] == "train"][-EMBARGO:]
print("Embargoed dates:", complete.loc[embargo_idx, "Date"].dt.strftime("%Y-%m-%d").tolist())
complete = complete.drop(index=embargo_idx)

# 8d. Final layout from the plan: date, y, f1-f10, split
model_table = (complete
               .rename(columns={"Date": "date"})
               [["date", "y"] + FEATURES + ["split"]]
               .reset_index(drop=True))

# 8e. Sanity checks
assert model_table[["y"] + FEATURES].notna().all().all(), "Missing values left"
assert model_table["date"].is_monotonic_increasing, "Dates out of order"
assert (model_table.loc[model_table["split"] == "train", "date"] <= TRAIN_END).all()
for split_name, part in model_table.groupby("split", sort=False):
    print(f"{split_name}: {len(part)} rows, {part['date'].min().date()} to {part['date'].max().date()}")

# 8f. Save the single table every model uses
os.makedirs("data/processed", exist_ok=True)
model_table.to_csv("data/processed/model_table.csv", index=False)

Dropped 27 incomplete rows: 22 at the start, 5 at the end
Embargoed dates: ['2014-12-24', '2014-12-26', '2014-12-29', '2014-12-30', '2014-12-31']
train: 2452 rows, 2005-03-30 to 2014-12-23
test: 717 rows, 2015-01-02 to 2017-11-03


# Step 9 — Set up CV splitter, metrics and prediction export

We define the shared tools every member uses, so all models are tuned and scored the same way. `cv_splitter()` returns TimeSeriesSplit(n_splits=5, gap=5), applied to training rows only: each fold fits on earlier data and scores on the next block, skipping 5 rows in between because each target looks 5 days ahead. `evaluate()` reports MAE, RMSE and R² on the log scale, plus QLIKE, which converts forecasts to variances (exp(2ŷ)) and penalises under-predicting risk more than over-predicting it. `save_predictions()` writes each model's test-period forecasts to `results/preds_<model>.csv` with columns date, y_true and y_pred; these files are the only hand-off between members.

In [35]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FEATURES = [f"f{i}" for i in range(1, 11)]   # f1 ... f10
N_SPLITS = 5                                  # number of CV folds (from the plan)
GAP = 5                                       # rows skipped between fit and score blocks

def load_model_table(path="data/processed/model_table.csv"):
    """Load the shared table from Step 8, with real dates."""
    return pd.read_csv(path, parse_dates=["date"])

# 9a. Cross-validation (CV) splitter, used on TRAINING rows only
def cv_splitter():
    """Time-series CV: 5 times, fit on earlier blocks and score on the next block.
    gap=5 skips 5 rows between them, because each target looks 5 days ahead (same reason as the embargo)."""
    return TimeSeriesSplit(n_splits=N_SPLITS, gap=GAP)

# 9b. QLIKE loss (Patton, 2011), computed on variances
def qlike(y_true, y_pred):
    """y is log volatility, so variance = exp(2y). Loss = ratio - log(ratio) - 1, with ratio = true / forecast.
    It is 0 for a perfect forecast and punishes under-prediction more than over-prediction."""
    true_var = np.exp(2 * np.asarray(y_true))
    pred_var = np.exp(2 * np.asarray(y_pred))   # conversion has a small bias (Jensen's inequality)
    ratio = true_var / pred_var
    return float(np.mean(ratio - np.log(ratio) - 1))

# 9c. evaluate(): the same four metrics for every model
def evaluate(y_true, y_pred):
    """MAE, RMSE and R² on the log scale, plus QLIKE. Lower is better, except R² (higher is better)."""
    return {
        "MAE":   float(mean_absolute_error(y_true, y_pred)),
        "RMSE":  float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2":    float(r2_score(y_true, y_pred)),
        "QLIKE": qlike(y_true, y_pred),
    }

# 9d. save_predictions(): the only hand-off between members
def save_predictions(model_name, test_table, y_pred, folder="results"):
    """Write results/preds_<model_name>.csv with columns date, y_true, y_pred (test period only)."""
    assert (test_table["split"] == "test").all(), "Pass only test-period rows"
    assert len(y_pred) == len(test_table), "Need exactly one forecast per test day"
    assert not np.isnan(np.asarray(y_pred)).any(), "Forecasts contain NaN"
    preds = pd.DataFrame({
        "date":   test_table["date"].dt.strftime("%Y-%m-%d").values,
        "y_true": test_table["y"].values,
        "y_pred": np.asarray(y_pred),
    })
    os.makedirs(folder, exist_ok=True)